In [2]:
import openai 
from openai import OpenAI
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import requests


/home/ethanliu/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("~/VibeCheck/data/arena_human_preference_train.csv")
df.head()

,id,model_name_1,model_name_2,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [4]:
import re
prompts = df["prompt"].tolist()[:10000]
prompts = [re.split(r"\?|\n", prompt)[0] + "?" if "?" in prompt else prompt for prompt in prompts]
prompts = [p if p.endswith("]") else p + "]" for p in prompts]
prompts = [p.strip("[]") for p in prompts]
print(len(prompts))
print(prompts[3353])

10000
"Give two reasons why clock frequency can increase if we change one load word into two cycles"


In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os  # Ensure os is imported

load_dotenv()  # Load environment variables from .env file
openai_api_key = os.getenv("OPENAI_API_KEY")  # Get the API key from environment variables

# Check if the API key is loaded correctly
if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY not found. Please set it in your environment or .env file.")

client = OpenAI(api_key=openai_api_key)  # Pass the API key to the client

def get_gpt4o_response(prompt):
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user", "content": prompt}]
    )
    return completion.choices[0].message.content

def get_gpt35_response(prompt):
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Changed to GPT-3.5
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user", "content": prompt}]
    )
    return completion.choices[0].message.content


In [6]:
import pandas as pd
import asyncio
import nest_asyncio
import os
from openai import OpenAI

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

client = OpenAI()

SAVE_PATH = "/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv"
batch_size = 20  # Adjust as needed

async def get_batch_gpt_responses(batch, model):
    async def fetch(prompt):
        try:
            response = await asyncio.to_thread(client.chat.completions.create, 
                model=model, 
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content 
        except Exception as e:
            return f"ERROR: {str(e)}"

    return await asyncio.gather(*[fetch(prompt) for prompt in batch])

async def main():
    data = []
    
    # Load existing data if the file exists
    if os.path.exists(SAVE_PATH):
        existing_df = pd.read_csv(SAVE_PATH)
        processed_prompts = set(existing_df["prompt"])
        print(f"Resuming from {len(existing_df)} saved responses.")
    else:
        existing_df = pd.DataFrame()
        processed_prompts = set()

    for i in range(0, len(prompts), batch_size):
        print(f"Processing batch {i // batch_size + 1}...")

        batch = prompts[i:i+batch_size]

        # Skip prompts that were already processed
        batch = [p for p in batch if p not in processed_prompts]
        if not batch:
            continue  # Skip batch if all prompts were already processed

        try:
            gpt4o_responses = await get_batch_gpt_responses(batch, "gpt-4o")
        except Exception as e:
            print(f"Error with GPT-4o batch {i}: {e}")
            gpt4o_responses = ["ERROR"] * len(batch)

        try:
            gpt35_responses = await get_batch_gpt_responses(batch, "gpt-3.5-turbo")
        except Exception as e:
            print(f"Error with GPT-3.5 batch {i}: {e}")
            gpt35_responses = ["ERROR"] * len(batch)

        for prompt, gpt4o, gpt35 in zip(batch, gpt4o_responses, gpt35_responses):
            data.append({"prompt": prompt, "gpt4o_response": gpt4o, "gpt35_response": gpt35})

        # Save after each batch
        batch_df = pd.DataFrame(data)
        if not existing_df.empty:
            batch_df = pd.concat([existing_df, batch_df], ignore_index=True)

        batch_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved {len(batch_df)} responses so far.")

    print("CSV file saved successfully!")

# Run event loop manually
loop = asyncio.get_event_loop()
loop.run_until_complete(main())

Processing batch 1...
Saved 20 responses so far.
Processing batch 2...
Saved 40 responses so far.
Processing batch 3...
Saved 60 responses so far.
Processing batch 4...
Saved 80 responses so far.
Processing batch 5...
Saved 100 responses so far.
Processing batch 6...
Saved 120 responses so far.
Processing batch 7...
Saved 140 responses so far.
Processing batch 8...
Saved 160 responses so far.
Processing batch 9...
Saved 180 responses so far.
Processing batch 10...
Saved 200 responses so far.
Processing batch 11...
Saved 220 responses so far.
Processing batch 12...
Saved 240 responses so far.
Processing batch 13...
Saved 260 responses so far.
Processing batch 14...
Saved 280 responses so far.
Processing batch 15...
Saved 300 responses so far.
Processing batch 16...
Saved 320 responses so far.
Processing batch 17...
Saved 340 responses so far.
Processing batch 18...
Saved 360 responses so far.
Processing batch 19...
Saved 380 responses so far.
Processing batch 20...
Saved 400 responses s

In [56]:
print(os.getcwd())
print(df.head())
df.to_csv("/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv", escapechar="\\", index=False)
print("CSV file saved successfully!")

/home/ethanliu/VibeCheck/notebooks
       id        model_name_1         model_name_2  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the process of invoking ...   
3  ["Creating a test set for a very rare category...   
4  ["The best way to tr

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import concurrent.futures
import logging
from tqdm import tqdm
from components.utils_general import get_emb_from_cache, save_emb_to_cache
from components.utils_llm import get_llm_embedding
import tiktoken

def normalize(vector):
    """Normalize vectors while avoiding division by zero"""
    norm = np.linalg.norm(vector, axis=1, keepdims=True)
    return np.where(norm == 0, vector, vector / norm)

In [16]:
import pickle
import pandas as pd
import numpy as np
import tiktoken
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def count_tokens(text, model="gpt-4o", max_tokens=8192):
    """Return the number of tokens in the text."""
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text, disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'}))
    return len(tokens)

def save_model(model, filename="mlp_classifier.pkl"):
    """Save the trained model to a file."""
    with open(filename, 'wb') as file:
        pickle.dump(model, file)
    print(f"Model saved to {filename}")

def load_model(filename="mlp_classifier.pkl"):
    """Load a trained model from a file."""
    with open(filename, 'rb') as file:
        model = pickle.load(file)
    print(f"Model loaded from {filename}")
    return model

def train_classifier_on_embeddings(dataset_path: str, max_rows: int = None):
    """
    Train MLP classifier on embeddings from two different models' outputs.
    Returns accuracy score and classifier.
    """
    # Load dataset
    print(f"Loading dataset from: {dataset_path}")
    df = pd.read_csv(dataset_path)
    if max_rows:
        df = df.head(max_rows)
    
    # Drop rows where responses exceed 8192 tokens
    df = df[
        (df["gpt4o_response"].apply(lambda x: count_tokens(str(x))) <= 8192) &
        (df["gpt35_response"].apply(lambda x: count_tokens(str(x))) <= 8192)
    ].reset_index(drop=True)

    print(f"Dataset reduced to {len(df)} rows after token limit filtering.")

    # Get embeddings
    print("Getting embeddings...")
    embeddings = {"gpt4o": [], "gpt35": []}
    
    counter = 0
    for col in ["gpt4o_response", "gpt35_response"]:
        for _, row in df.iterrows():
            formatted_text = f"Response:{row[col]}"
            emb = get_llm_embedding(formatted_text, "text-embedding-ada-002")
            if emb is None:
                print(f"Failed to get embedding for text in column {col}")
                return None, None
            embeddings[col.split("_")[0]].append(emb)
            counter += 1
    
    print("Length of col and row in df: ", counter)
    
    df["gpt4o_embedding"] = embeddings["gpt4o"]
    df["gpt35_embedding"] = embeddings["gpt35"]
    
    # Convert embeddings to numpy arrays
    try:
        gpt4o_embeddings = np.vstack(df['gpt4o_embedding'].values)
        gpt35_embeddings = np.vstack(df['gpt35_embedding'].values)
    except Exception as e:
        print(f"Error converting embeddings to numpy arrays: {e}")
        return None, None
    
    # Compute normalized differences
    print("Computing normalized differences...")
    diff = normalize(gpt4o_embeddings - gpt35_embeddings)  # (GPT-4o - GPT-3.5)
    inv_diff = normalize(gpt35_embeddings - gpt4o_embeddings)  # (GPT-3.5 - GPT-4o)
    
    # Create labels
    y_diff = np.ones(len(diff))  # Label 1 for (gpt4o - gpt3.5)
    y_inv_diff = np.zeros(len(inv_diff))  # Label 0 for (gpt3.5 - gpt4o)
    
    # Stack features and labels to balance dataset
    X = np.vstack([diff, inv_diff])  # Shape: (2*n_samples, __)
    y = np.hstack([y_diff, y_inv_diff])  # Shape: (2*n_samples,)
    
    print(f"Feature matrix shape: {X.shape}")
    print(f"Labels shape: {y.shape}")
    
    # Split data
    print("Training classifier...")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize and train MLP
    clf = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=1000, random_state=42, early_stopping=True, verbose=True)
    clf.fit(X_train, y_train)
    
    print("NP bincount:", np.bincount(y_train.astype(int)))
    
    # Get predictions
    y_pred = clf.predict(X_test)
    
    # Compute evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    print("Classification Metrics:")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

    # Save model
    save_model(clf, "mlp_classifier.pkl")
    
    return accuracy, clf

if __name__ == "__main__":
    dataset_path = "/home/ethanliu/VibeCheck/notebooks/model_responses.csv"
    accuracy, classifier = train_classifier_on_embeddings(dataset_path, max_rows=10000)
    if accuracy is not None:
        print(f"Final accuracy: {accuracy}")
    else:
        print("Failed to train classifier due to embedding errors")

Loading dataset from: /home/ethanliu/VibeCheck/notebooks/model_responses.csv
Dataset reduced to 9999 rows after token limit filtering.
Getting embeddings...
Length of col and row in df:  19998
Computing normalized differences...
Feature matrix shape: (19998, 1536)
Labels shape: (19998,)
Training classifier...
Iteration 1, loss = 0.28195446
Validation score: 0.943750
Iteration 2, loss = 0.10366644
Validation score: 0.950000
Iteration 3, loss = 0.07237877
Validation score: 0.951875
Iteration 4, loss = 0.05208761
Validation score: 0.952500
Iteration 5, loss = 0.03529736
Validation score: 0.946875
Iteration 6, loss = 0.02293783
Validation score: 0.951875
Iteration 7, loss = 0.01583351
Validation score: 0.951875
Iteration 8, loss = 0.01209805
Validation score: 0.951875
Iteration 9, loss = 0.01151052
Validation score: 0.952500
Iteration 10, loss = 0.01115248
Validation score: 0.951875
Iteration 11, loss = 0.01056582
Validation score: 0.951250
Iteration 12, loss = 0.01087943
Validation score:

In [94]:
if __name__ == "__main__":
    dataset_path = "/home/ethanliu/VibeCheck/notebooks/model_responses.csv"
    accuracy, classifier = train_classifier_on_embeddings(dataset_path, max_rows=10000)
    if accuracy is not None:
        print(f"Final accuracy: {accuracy}")
    else:
        print("Failed to train classifier due to embedding errors")

Loading dataset from: /home/ethanliu/VibeCheck/notebooks/model_responses.csv
Dataset reduced to 9999 rows after token limit filtering.
Getting embeddings...
Length of col and row in df:  19998
Computing normalized differences...
Feature matrix shape: (19998, 1536)
Labels shape: (19998,)
Training classifier...
Iteration 1, loss = 0.28195446
Validation score: 0.943750
Iteration 2, loss = 0.10366644
Validation score: 0.950000
Iteration 3, loss = 0.07237877
Validation score: 0.951875
Iteration 4, loss = 0.05208761
Validation score: 0.952500
Iteration 5, loss = 0.03529736
Validation score: 0.946875
Iteration 6, loss = 0.02293783
Validation score: 0.951875
Iteration 7, loss = 0.01583351
Validation score: 0.951875
Iteration 8, loss = 0.01209805
Validation score: 0.951875
Iteration 9, loss = 0.01151052
Validation score: 0.952500
Iteration 10, loss = 0.01115248
Validation score: 0.951875
Iteration 11, loss = 0.01056582
Validation score: 0.951250
Iteration 12, loss = 0.01087943
Validation score:

In [1]:
import pandas as pd
df = pd.read_csv("/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv")
df_prompts = df.sample(n=5, random_state=40)['prompt']
df_response_4o = df.sample(n=5, random_state=40)['gpt4o_response']
print(df_prompts)
print(df_response_4o.iloc[0])

293     "Why are people so much obsessed with gender a...
1244    "Write a contextual exegesis for \"Seek and ye...
7353    "Where is the letter r in the word \"blueberry\""
5145    "Label this list of companies based on their s...
1618    "15-Word Text about apes. All words should onl...
Name: prompt, dtype: object
People's focus on gender and racism often stems from the historical and ongoing impacts these issues have on society. Here are a few reasons why they receive significant attention:

1. **Historical Context**: Both gender and racism have deep roots in history and have influenced social structures, power dynamics, and inequalities for centuries. Discussions about these topics seek to address and rectify historical injustices.

2. **Inequality and Discrimination**: Gender and racial inequalities persist in many areas, including the workplace, education, healthcare, and beyond. People are focused on these issues to advocate for equal rights and fair treatment for all individual

In [2]:
# System prompt for gpt 3.5 to act like gpt-4o 
system_prompt = f'''You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: {df_prompts.iloc[0]}
response: {df_response_4o.iloc[0]}

Example 2:
prompt: {df_prompts.iloc[1]}
response: {df_response_4o.iloc[1]}

Example 3:
prompt: {df_prompts.iloc[2]}
response: {df_response_4o.iloc[2]}

Example 4:
prompt: {df_prompts.iloc[3]}
response: {df_response_4o.iloc[3]}

Example 5:
prompt: {df_prompts.iloc[4]}
response: {df_response_4o.iloc[4]}

Here is the question to answer: '''

print(system_prompt)

You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: "Why are people so much obsessed with gender and racism?
response: People's focus on gender and racism often stems from the historical and ongoing impacts these issues have on society. Here are a few reasons why they receive significant attention:

1. **Historical Context**: Both gender and racism have deep roots in history and have influenced social structures, power dynamics, and inequalities for centuries. Discussions about these topics seek to address and rectify historical injustices.

2. **Inequality and Discrimination**: Gender and racial inequalities persist in many areas, including the workplace, education, healthcare, and beyond. People are focused on these issues to advocate for equal rights and fair treatment for all individuals, irrespective of their race or 

In [4]:
import pandas as pd
import asyncio
import nest_asyncio
import os
from openai import OpenAI
import tiktoken  # For token counting
from dotenv import load_dotenv

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

load_dotenv()  # Load environment variables from .env file
openai_api_key = os.getenv("OPENAI_API_KEY")  # Get the API key from environment variables

client = OpenAI()
encoder = tiktoken.encoding_for_model("gpt-3.5-turbo")

SAVE_PATH = "/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv"
batch_size = 20  # Adjust as needed
TOKEN_LIMIT = 8192  # Max token limit for GPT-3.5-turbo

# Load dataset
df = pd.read_csv(SAVE_PATH)
total_rows = len(df)
print(f"Total rows in dataset: {total_rows}")
assert total_rows == 10000, f"Expected 10000 rows, but found {total_rows}"

async def get_batch_gpt_responses(batch):
    async def fetch(prompt):
        full_prompt = system_prompt + "\n" + prompt

        # Token check
        num_tokens = len(encoder.encode(full_prompt))
        if num_tokens > TOKEN_LIMIT:
            return "N/A"  # Skip this prompt if it's too long

        try:
            response = await asyncio.to_thread(client.chat.completions.create, 
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": full_prompt}]
            )
            return response.choices[0].message.content 
        except Exception as e:
            return f"ERROR: {str(e)}"

    return await asyncio.gather(*[fetch(prompt) for prompt in batch])

async def main():
    # Check if "gpt35_reprompted" column exists, else add it
    if "gpt35_reprompted" not in df.columns:
        df["gpt35_reprompted"] = pd.NA
        
    # Reset all existing responses to track new processing
    df["gpt35_reprompted"] = pd.NA
    processed_count = 0

    prompts = df["prompt"].tolist()
    total_batches = (len(prompts) + batch_size - 1) // batch_size
    
    print(f"Starting processing of {len(prompts)} prompts in {total_batches} batches")
    
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        print(f"Processing batch {i // batch_size + 1}/{total_batches}...")        

        try: 
            responses = await get_batch_gpt_responses(batch)
            for j, prompt in enumerate(batch):
                df.loc[df["prompt"] == prompt, "gpt35_reprompted"] = responses[j]
                processed_count += 1
            
            df.to_csv(SAVE_PATH, index=False)
            print(f"Processed {processed_count}/{total_rows} responses")

        except Exception as e:
            print(f"Error in batch {i // batch_size + 1}: {e}")
            continue
            
    print(f"Processing complete. Total responses: {processed_count}/{total_rows}")
    assert processed_count == total_rows, f"Expected {total_rows} responses, but got {processed_count}"

# Run event loop
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main())

Total rows in dataset: 10000
Starting processing of 10000 prompts in 500 batches
Processing batch 1/500...
Processed 20/10000 responses
Processing batch 2/500...
Processed 40/10000 responses
Processing batch 3/500...
Processed 60/10000 responses
Processing batch 4/500...
Processed 80/10000 responses
Processing batch 5/500...
Processed 100/10000 responses
Processing batch 6/500...
Processed 120/10000 responses
Processing batch 7/500...
Processed 140/10000 responses
Processing batch 8/500...
Processed 160/10000 responses
Processing batch 9/500...
Processed 180/10000 responses
Processing batch 10/500...
Processed 200/10000 responses
Processing batch 11/500...
Processed 220/10000 responses
Processing batch 12/500...
Processed 240/10000 responses
Processing batch 13/500...
Processed 260/10000 responses
Processing batch 14/500...
Processed 280/10000 responses
Processing batch 15/500...
Processed 300/10000 responses
Processing batch 16/500...
Processed 320/10000 responses
Processing batch 17/

In [5]:
import pandas as pd

SAVE_PATH = "/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv"

# Load dataset
df = pd.read_csv(SAVE_PATH)

# Check for empty or NaN values in `gpt35_reprompted`
print("Total rows:", len(df))
print("Rows where gpt35_reprompted is NaN:", df["gpt35_reprompted"].isna().sum())
print("Rows where gpt35_reprompted is an empty string:", (df["gpt35_reprompted"] == "").sum())

# Show first few rows to manually inspect values
print(df[["prompt", "gpt35_reprompted"]].head(10))

Total rows: 10000
Rows where gpt35_reprompted is NaN: 0
Rows where gpt35_reprompted is an empty string: 0
                                              prompt  \
0  "Is it morally right to try to have a certain ...   
1  "What is the difference between marriage licen...   
2  "explain function calling. how would you call ...   
3  "How can I create a test set for a very rare c...   
4  "What is the best way to travel from Tel-Aviv ...   
5  "Construct a rap battle, in the style of Epic ...   
6                "Why water is not used in bath tub?   
7  "\"Bacteria is life on Mars but a heartbeat is...   
8  "translate to russian the followig sentence  B...   
9  "From now, you *always* have to talk as if you...   

                                    gpt35_reprompted  
0  Having a certain percentage of females in mana...  
1  The difference between a marriage license and ...  
2  Function calling is a fundamental concept in p...  
3  Creating a test set for a very rare category r...  
4 

In [8]:
import pickle
import pandas as pd
import numpy as np
import tiktoken
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from components.utils_llm import get_llm_embedding

# Load the trained model
def load_model(filename="/home/ethanliu/VibeCheck/notebooks/mlp_classifier.pkl"):
    """Load a trained model from a file."""
    with open(filename, 'rb') as file:
        model = pickle.load(file)
    print(f"Model loaded from {filename}")
    return model

# Load dataset
dataset_path = "/home/ethanliu/VibeCheck/notebooks/disguising/model_responses.csv"
df = pd.read_csv(dataset_path)

# Filter rows where gpt35_reprompted is not null
df = df[df["gpt35_reprompted"].notna()]

# Sample 2000 rows randomly
df_sampled = df.sample(n=2000, random_state=42).reset_index(drop=True)

# Function to count tokens (needed for filtering if required)
def count_tokens(text, model="gpt-4o", max_tokens=8192):
    """Return the number of tokens in the text."""
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(str(text), disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'}))
    return len(tokens)

# Ensure responses do not exceed 8192 tokens (if needed)
df_sampled = df_sampled[
    (df_sampled["gpt4o_response"].apply(lambda x: count_tokens(str(x))) <= 8192) &
    (df_sampled["gpt35_reprompted"].apply(lambda x: count_tokens(str(x))) <= 8192)
].reset_index(drop=True)

print(f"Sampled dataset reduced to {len(df_sampled)} rows after token limit filtering.")

# Function to get embeddings
def get_embedding(text, model="text-embedding-ada-002"):
    """Retrieve embedding for a given text (placeholder function)."""
    # Assuming you have a function like get_llm_embedding(text, model)
    return get_llm_embedding(text, model)

# Compute embeddings
print("Computing embeddings for sampled dataset...")
embeddings = {"gpt4o": [], "gpt35_reprompted": []}

for col in ["gpt4o_response", "gpt35_reprompted"]:
    for _, row in df_sampled.iterrows():
        formatted_text = f"Response:{row[col]}"
        emb = get_embedding(formatted_text, "text-embedding-ada-002")
        if emb is None:
            print(f"Failed to get embedding for text in column {col}")
            exit()
        key = "gpt4o" if col == "gpt4o_response" else "gpt35_reprompted"
        embeddings[key].append(emb)

df_sampled["gpt4o_embedding"] = embeddings["gpt4o"]
df_sampled["gpt35_reprompted_embedding"] = embeddings["gpt35_reprompted"]

# Convert embeddings to numpy arrays
try:
    gpt4o_embeddings = np.vstack(df_sampled['gpt4o_embedding'].values)
    gpt35_embeddings = np.vstack(df_sampled['gpt35_reprompted_embedding'].values)
except Exception as e:
    print(f"Error converting embeddings to numpy arrays: {e}")
    exit()

# Compute normalized differences
X_test = normalize(gpt4o_embeddings - gpt35_embeddings)

# Generate true labels (y_true) based on the difference between embeddings
# Assign 1 if GPT-4o's embedding is greater than GPT-3.5's embedding, else assign 0
y_true = np.array([1 if np.linalg.norm(gpt4o - gpt35) > 0 else 0
                   for gpt4o, gpt35 in zip(gpt4o_embeddings, gpt35_embeddings)])

# Here we use `y_true` as both the ground truth (y_test) and our label for performance evaluation
y_test = y_true

# Load trained model
clf = load_model("mlp_classifier.pkl")

# Make predictions
y_pred = clf.predict(X_test)

# Compute evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')
conf_matrix = confusion_matrix(y_test, y_pred)

# Print the metrics
print("Classification Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

# Print results
print(f"Predictions on 2000 sampled rows: {y_pred[:20]}")  # Print first 20 predictions for preview
print(f"Class distribution in predictions: {np.bincount(y_pred.astype(int))}")

Sampled dataset reduced to 2000 rows after token limit filtering.
Computing embeddings for sampled dataset...
Model loaded from mlp_classifier.pkl
Classification Metrics:
Accuracy:  0.8680
Precision: 0.9943
Recall:    0.8724
F1 Score:  0.9293
Confusion Matrix:
[[   0   10]
 [ 254 1736]]
Predictions on 2000 sampled rows: [1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Class distribution in predictions: [ 254 1746]


In [16]:
# Check if the original dataframe is still in memory
import gc
for obj in gc.get_objects():
    if isinstance(obj, pd.DataFrame) and len(obj) == 10000:
        recovered_df = obj.copy()
        print("Found DataFrame with 10000 rows!")
        recovered_df.to_csv(SAVE_PATH, index=False)
        break